In [49]:
import joblib
import pandas as pd

In [50]:
model = joblib.load("../models/gradient_boosting_model.pkl")
scaler = joblib.load("../models/scaler.pkl")

In [51]:
SCALER_FEATURES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Type_H",
    "Type_L",
    "Type_M",
]

MODEL_FEATURES = [
    "Type_H",
    "Type_L",
    "Type_M",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

In [52]:
def predict_machine(
    machine_type,
    air_temperature,
    process_temperature,
    rotational_speed,
    torque,
    tool_wear,
):

    machine = pd.DataFrame([{
        "Air temperature [K]": air_temperature,
        "Process temperature [K]": process_temperature,
        "Rotational speed [rpm]": rotational_speed,
        "Torque [Nm]": torque,
        "Tool wear [min]": tool_wear,
        "Type_H": int(machine_type == "H"),
        "Type_L": int(machine_type == "L"),
        "Type_M": int(machine_type == "M"),
    }], columns=SCALER_FEATURES)

    scaled_machine = scaler.transform(machine)

    scaled_machine = pd.DataFrame(
        scaled_machine,
        columns=SCALER_FEATURES
    )

    scaled_machine = scaled_machine[MODEL_FEATURES]
    probability = model.predict_proba(scaled_machine)[0]
    prediction = int(probability[1] >= 0.35)

    return {
        "prediction": int(prediction),
        "probability_no_failure": float(probability[0]),
        "probability_failure": float(probability[1]),
    }

In [55]:
result = predict_machine(
    machine_type="M",
    air_temperature=298.1,
    process_temperature=308.6,
    rotational_speed=1551,
    torque=42.8,
    tool_wear=0
)

print(result)

{'prediction': 0, 'probability_no_failure': 0.9977184498198264, 'probability_failure': 0.002281550180173653}


In [54]:
if result["prediction"] == 1:
    print("Machine failure predicted")
else:
    print("No machine failure predicted")

print(
    f"Failure probability: "
    f"{result['probability_failure'] * 100:.2f}%"
)

No machine failure predicted
Failure probability: 0.23%
